# Medication dosing per patient sex
Key question: does dosing of medication (fentanyl & ketamine) differ by sex of the patient?

In [ ]:
import pandas as pd
import re
import numpy as np

In [ ]:
data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/data/rega_data/trauma_categories_Rega Pain Study15.09.2025_v2.xlsx'
medic_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/data/rega_data/rega_physician_list_09102025.xlsx'
meta_medic_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/data/medreg_extraction/joined_final_complete_extractions_20251008_221735.xlsx'

In [ ]:
restrict_to_trauma = True
restrict_to_primary = True

In [ ]:
data_df = pd.read_excel(data_path)
medic_df = pd.read_excel(medic_data_path)
meta_medic_df = pd.read_excel(meta_medic_data_path)

In [ ]:
medic_df['full_name'] = medic_df['Mitglieder mit Einsatzfunktion'].str.replace(' (Flugarzt/Flugärztin)', '')
medic_df.drop_duplicates(subset=['Mitglieder mit Einsatzfunktion'], inplace=True)
medic_df = medic_df.merge(meta_medic_df, how='left', on='full_name')
medic_df.rename(columns={'Sex m/w': 'physician_sex'}, inplace=True)
data_df = data_df.merge(medic_df, how='left', left_on='Mitglieder mit Einsatzfunktion', right_on='Mitglieder mit Einsatzfunktion')

In [ ]:
duplicates = data_df[data_df["SNZ Ereignis Nr. "].duplicated()]["SNZ Ereignis Nr. "]
print(f'Duplicates found: {duplicates.nunique()}')
# drop duplicates
data_df = data_df.drop_duplicates(subset=["SNZ Ereignis Nr. "])

In [ ]:
n_vas_under4 = data_df[data_df["VAS_on_scene"] <= 3].shape[0]
print(f'Excluded {n_vas_under4} patients with VAS <= 3')

# adult patients with vas <= 3
n_adult_vas_under4 = data_df[(data_df["VAS_on_scene"] <= 3) & (data_df["Alter "] >= 16)].shape[0]
print(f'Excluded {n_adult_vas_under4} adult patients with VAS <= 3')

# pediatric patients with vas <= 3
n_pediatric_vas_under4 = data_df[(data_df["VAS_on_scene"] <= 3) & (data_df["Alter "] < 16)].shape[0]
print(f'Excluded {n_pediatric_vas_under4} pediatric patients with VAS <= 3')

data_df = data_df[data_df["VAS_on_scene"] > 3]

In [ ]:
if restrict_to_trauma:
    n_non_trauma = data_df[data_df['Einteilung (reduziert)'] != 'Unfall'].shape[0]
    print(f'Excluded {n_non_trauma} non-trauma patients')

    # adult non-trauma patients
    n_adult_non_trauma = data_df[(data_df['Einteilung (reduziert)'] != 'Unfall') & (data_df["Alter "] >= 16)].shape[0]
    print(f'Excluded {n_adult_non_trauma} adult non-trauma patients')
    # pediatric non-trauma patients
    n_pediatric_non_trauma = data_df[(data_df['Einteilung (reduziert)'] != 'Unfall') & (data_df["Alter "] < 16)].shape[0]
    print(f'Excluded {n_pediatric_non_trauma} pediatric non-trauma patients')

    data_df = data_df[data_df['Einteilung (reduziert)'] == 'Unfall']

In [ ]:
if restrict_to_primary:
    n_secondary = data_df[data_df['Einsatzart'] != 'Primär'].shape[0]
    print(f'Excluded {n_secondary} secondary transport patients')

    # adult secondary transport patients
    n_adult_secondary = data_df[(data_df['Einsatzart'] != 'Primär') & (data_df["Alter "] >= 16)].shape[0]
    print(f'Excluded {n_adult_secondary} adult secondary transport patients')
    # pediatric secondary transport patients
    n_pediatric_secondary = data_df[(data_df['Einsatzart'] != 'Primär') & (data_df["Alter "] < 16)].shape[0]
    print(f'Excluded {n_pediatric_secondary} pediatric secondary transport patients')
    data_df = data_df[data_df['Einsatzart'] == 'Primär']


In [ ]:
adult_df = data_df[data_df["Alter "] >= 16]
pediatric_df = data_df[data_df["Alter "] < 16]

In [ ]:
adult_df = adult_df[~adult_df['VAS_on_arrival'].isna()]

In [ ]:
len(adult_df)

In [ ]:
# Create medication dose variables (matching Table 1 approach)
adult_df['fentanyl_dose'] = 0
adult_df['ketamine_dose'] = 0
adult_df['esketamine_dose'] = 0
adult_df['morphine_dose'] = 0
adult_df['Alle Medikamente'] = adult_df['Alle Medikamente'].str.replace(',', ';')  # replace commas with semicolons for consistency
for i, row in adult_df.iterrows():
    if pd.isna(row['Alle Medikamente']) or row['Alle Medikamente'] == 0:
        continue
    for analgetic in row['Alle Medikamente'].split(';'):
        if analgetic.strip() == '':
            continue
        # remove mcg or mg from dose
        if '7IE' in analgetic:
                print(f"Skipping dose with 7IE: {analgetic}")
                continue

        analgetic = analgetic.replace('mcg', '').replace('mg', '').strip()
        if 'Fentanyl' in analgetic and '/h' not in analgetic:
            dose = analgetic.split('Fentanyl')[-1].strip()
            adult_df.at[i, 'fentanyl_dose'] += float(dose) 
        elif 'Fentanyl' in analgetic and '/h' in analgetic:
            dose = analgetic.split('Fentanyl')[-1].strip().replace('/h', '')
            dose = float(dose) * adult_df.at[i, 'mission_duration']  
            adult_df.at[i, 'fentanyl_dose'] += float(dose)
        elif 'Ketamin' in analgetic or 'Ketamine' in analgetic:
            dose = analgetic.split('Ketamin')[-1].strip()
            adult_df.at[i, 'ketamine_dose'] += float(dose)
        elif 'Esketamin' in analgetic:
            dose = analgetic.split('Esketamin')[-1].strip()
            adult_df.at[i, 'esketamine_dose'] += float(dose)
        elif 'Morphin' in analgetic or 'Morphine' in analgetic:
            dose = analgetic.split('Morphin')[-1].strip()
            adult_df.at[i, 'morphine_dose'] += float(dose)

# Create medication variables
adult_df['fentanyl_given'] = adult_df['fentanyl_dose'] > 0
adult_df['morphine_given'] = adult_df['morphine_dose'] > 0
adult_df['ketamine_given'] = adult_df['ketamine_dose'] > 0
adult_df['esketamine_given'] = adult_df['esketamine_dose'] > 0

# Create combined medication variables (PRIMARY VARIABLES OF INTEREST)
adult_df['any_opiate_dose'] = adult_df['morphine_dose'] + adult_df['fentanyl_dose']
adult_df['any_ketamine_dose'] = adult_df['ketamine_dose'] + adult_df['esketamine_dose']
adult_df['any_opiate_given'] = (adult_df['morphine_dose'] > 0) | (adult_df['fentanyl_dose'] > 0)
adult_df['any_ketamine_given'] = (adult_df['ketamine_dose'] > 0) | (adult_df['esketamine_dose'] > 0)


In [ ]:
adult_df.Geschlecht.value_counts()

In [ ]:
# create box/violinplots with patient sex on x axis and medication dose on y axis, split by subgroup (any_opiate, any_ketamine)
import seaborn as sns
import matplotlib.pyplot as plt

plot_type = 'violin'


In [ ]:
plot_df = adult_df.copy()
plot_df["patient_sex"] = plot_df["Geschlecht"].replace({
    "m": "Male",
    "w": "Female",
    "M": "Male",
    "W": "Female",
    "male": "Male",
    "female": "Female",
    "Männlich": "Male",
    "Weiblich": "Female",
})
plot_df = plot_df[plot_df["patient_sex"].isin(["Male", "Female"])]

dose_long = plot_df.melt(
    id_vars=["patient_sex"],
    value_vars=["any_opiate_dose", "any_ketamine_dose"],
    var_name="medication_group",
    value_name="dose",
)
dose_long = dose_long[dose_long["dose"].notna()]
dose_long["medication_group"] = dose_long["medication_group"].replace({
    "any_opiate_dose": "Any opiate",
    "any_ketamine_dose": "Any ketamine",
})

dose_long_plot = dose_long[dose_long["dose"] >= 0]

# Option to remove extremes above 99th percentile (per medication group)
remove_extremes = True
if remove_extremes:
    p99 = dose_long_plot.groupby("medication_group")["dose"].quantile(0.99)
    dose_long_plot = dose_long_plot[dose_long_plot.apply(
        lambda r: r["dose"] <= p99.loc[r["medication_group"]], axis=1
    )]

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True)
order = ["Male", "Female"]

opiate_df = dose_long_plot[dose_long_plot["medication_group"] == "Any opiate"]
ketamine_df = dose_long_plot[dose_long_plot["medication_group"] == "Any ketamine"]

sns.violinplot(
    data=opiate_df,
    x="patient_sex",
    y="dose",
    order=order,
    hue="patient_sex",
    ax=axes[0],
    cut=0,
)
axes[0].set_xlabel("")
axes[0].set_ylabel("Opiate dose (mcg)")
if axes[0].legend_ is not None:
    axes[0].legend_.remove()
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)
axes[0].yaxis.grid(True, linestyle="--", alpha=0.4)

sns.violinplot(
    data=ketamine_df,
    x="patient_sex",
    y="dose",
    order=order,
    hue="patient_sex",
    ax=axes[1],
    cut=0,
)
axes[1].set_xlabel("")
axes[1].set_ylabel("Ketamine dose (mg)")
if axes[1].legend_ is not None:
    axes[1].legend_.remove()
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
axes[1].yaxis.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# fig.savefig('/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/adult_trauma_analgesia/review2/dosing_per_patient_sex_violin.png', dpi=300)

In [ ]:
# plot with excluding 0 doses
# create box/violin plots with patient sex on x axis and medication dose on y axis, split by subgroup (any_opiate, any_ketamine)


In [ ]:
plot_df = adult_df.copy()
plot_df["patient_sex"] = plot_df["Geschlecht"].replace({
    "m": "Male",
    "w": "Female",
    "M": "Male",
    "W": "Female",
    "male": "Male",
    "female": "Female",
    "Männlich": "Male",
    "Weiblich": "Female",
})
plot_df = plot_df[plot_df["patient_sex"].isin(["Male", "Female"])]

dose_long = plot_df.melt(
    id_vars=["patient_sex"],
    value_vars=["any_opiate_dose", "any_ketamine_dose"],
    var_name="medication_group",
    value_name="dose",
)
dose_long = dose_long[dose_long["dose"].notna()]
dose_long["medication_group"] = dose_long["medication_group"].replace({
    "any_opiate_dose": "Any opiate",
    "any_ketamine_dose": "Any ketamine",
})

dose_long_plot = dose_long[dose_long["dose"] > 0]

# Option to remove extremes above 99th percentile (per medication group)
remove_extremes = True
if remove_extremes:
    p99 = dose_long_plot.groupby("medication_group")["dose"].quantile(0.99)
    dose_long_plot = dose_long_plot[dose_long_plot.apply(
        lambda r: r["dose"] <= p99.loc[r["medication_group"]], axis=1
    )]

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True)
order = ["Male", "Female"]

opiate_df = dose_long_plot[dose_long_plot["medication_group"] == "Any opiate"]
ketamine_df = dose_long_plot[dose_long_plot["medication_group"] == "Any ketamine"]

sns.violinplot(
    data=opiate_df,
    x="patient_sex",
    y="dose",
    order=order,
    hue="patient_sex",
    ax=axes[0],
    cut=0,
)
axes[0].set_xlabel("")
axes[0].set_ylabel("Opiate dose (mcg)")
if axes[0].legend_ is not None:
    axes[0].legend_.remove()
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)
axes[0].yaxis.grid(True, linestyle="--", alpha=0.4)

sns.violinplot(
    data=ketamine_df,
    x="patient_sex",
    y="dose",
    order=order,
    hue="patient_sex",
    ax=axes[1],
    cut=0,
)
axes[1].set_xlabel("")
axes[1].set_ylabel("Ketamine dose (mg)")
if axes[1].legend_ is not None:
    axes[1].legend_.remove()
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
axes[1].yaxis.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Statistical comparison of medication dose between patient sex categories
from scipy.stats import mannwhitneyu, norm, rankdata

def mannwhitney_p_display(male_values, female_values):
    """Return U, p-value, and a robust p-value display that avoids underflow to 0."""
    n1, n2 = len(male_values), len(female_values)
    if n1 == 0 or n2 == 0:
        return np.nan, np.nan, "NA"

    u_stat, p_value = mannwhitneyu(male_values, female_values, alternative="two-sided")

    # If p underflows to 0, compute log-p with tie-corrected normal approximation and display bound
    if p_value == 0:
        values = np.concatenate([male_values.to_numpy(), female_values.to_numpy()])
        ranks = rankdata(values, method="average")
        rank_male_sum = ranks[:n1].sum()
        u1 = rank_male_sum - n1 * (n1 + 1) / 2
        mean_u = n1 * n2 / 2

        _, counts = np.unique(values, return_counts=True)
        tie_term = np.sum(counts**3 - counts)
        n = n1 + n2
        var_u = n1 * n2 / 12 * ((n + 1) - tie_term / (n * (n - 1)))

        if var_u <= 0:
            return u_stat, p_value, "0"

        z = (u1 - mean_u) / np.sqrt(var_u)
        log_p_two_sided = np.log(2) + norm.logsf(np.abs(z))
        log10_p = log_p_two_sided / np.log(10)
        return u_stat, p_value, f"< 1e{int(np.floor(log10_p))}"

    return u_stat, p_value, f"{p_value:.3e}"

stats_df = adult_df.copy()
sex_raw = stats_df["Geschlecht"].astype(str).str.strip().str.lower()
stats_df["patient_sex"] = np.nan
stats_df.loc[sex_raw.str.startswith("m"), "patient_sex"] = "Male"
stats_df.loc[sex_raw.str.startswith("w") | sex_raw.str.startswith("f"), "patient_sex"] = "Female"
stats_df = stats_df[stats_df["patient_sex"].isin(["Male", "Female"])]

medication_vars = [
    ("any_opiate_dose", "Any opiate dose (mcg)"),
    ("any_ketamine_dose", "Any ketamine dose (mg)"),
]

rows = []
for med_col, med_label in medication_vars:
    for zero_rule, comparator in [("including_zeros", lambda s: s >= 0), ("excluding_zeros", lambda s: s > 0)]:
        male = stats_df.loc[stats_df["patient_sex"] == "Male", med_col].dropna()
        female = stats_df.loc[stats_df["patient_sex"] == "Female", med_col].dropna()

        male = male[comparator(male)]
        female = female[comparator(female)]

        if len(male) > 0 and len(female) > 0:
            u_stat, p_value, p_display = mannwhitney_p_display(male, female)
        else:
            u_stat, p_value, p_display = np.nan, np.nan, "NA"

        male_q1 = male.quantile(0.25) if len(male) else np.nan
        male_q3 = male.quantile(0.75) if len(male) else np.nan
        female_q1 = female.quantile(0.25) if len(female) else np.nan
        female_q3 = female.quantile(0.75) if len(female) else np.nan

        rows.append({
            "medication": med_label,
            "analysis": zero_rule,
            "male_n": len(male),
            "female_n": len(female),
            "male_median": male.median() if len(male) else np.nan,
            "female_median": female.median() if len(female) else np.nan,
            "male_q1": male_q1,
            "male_q3": male_q3,
            "female_q1": female_q1,
            "female_q3": female_q3,
            "male_iqr_width": (male_q3 - male_q1) if len(male) else np.nan,
            "female_iqr_width": (female_q3 - female_q1) if len(female) else np.nan,
            "mannwhitney_u": u_stat,
            "p_value": p_value,
            "p_value_display": p_display,
        })

stats_results = pd.DataFrame(rows)
stats_results["p_value_bonferroni_4tests"] = (stats_results["p_value"] * 4).clip(upper=1.0)
stats_results["p_value_bonferroni_display"] = stats_results["p_value_bonferroni_4tests"].apply(
    lambda p: "NA" if pd.isna(p) else (f"{p:.3e}" if p > 0 else "< machine precision")
)

display(stats_results)